# Deep Learning for State Poverty Rate Prediction

**Objective:** Apply deep learning methods to predict US state poverty rates using socio-economic predictors.

**Methodology:** Following the approach from the Somalia poverty study (Nature Scientific Reports 2024), we implement:
- Neural Networks with keras
- Random Forest for comparison
- Feature importance analysis
- Model performance evaluation (RMSE, R², MAE)

**Data:** State-level cross-sectional data (n=50) for 2023

## 1. Setup and Data Loading

In [ ]:
# Install required packages
install.packages(c("keras3", "tidyverse", "randomForest", "caret", "MLmetrics", "gridExtra"))

library(keras3)
library(tidyverse)
library(randomForest)
library(caret)
library(MLmetrics)

In [ ]:
# Configure Keras backend (automatically uses TensorFlow)
# No additional installation needed with keras3

In [ ]:
# Load processed state data
# For local use: make sure file is in data/processed/dataset.csv
# For Colab: upload file or mount Google Drive, then adjust path below

# Try multiple possible paths
if (file.exists("data/processed/dataset.csv")) {
  data <- read_csv("data/processed/dataset.csv")
} else if (file.exists("dataset.csv")) {
  data <- read_csv("dataset.csv")
} else {
  stop("Data file not found. Please upload dataset.csv or check the path.")
}

# Display structure
glimpse(data)
summary(data)

## 2. Data Preparation

In [ ]:
# Remove State column (keep only numeric predictors and outcome)
# Outcome variable is 'poverty'
data_clean <- data %>% 
  select(-State) %>%
  na.omit()

# Check final dimensions
cat("Final dataset:", nrow(data_clean), "observations,", ncol(data_clean), "variables\n")
cat("Outcome variable: poverty\n")
cat("Number of predictors:", ncol(data_clean) - 1, "\n")

In [ ]:
# Train-test split (80-20)
set.seed(2026)
train_idx <- sample(1:nrow(data_clean), size = 0.8 * nrow(data_clean))

train_data <- data_clean[train_idx, ]
test_data <- data_clean[-train_idx, ]

cat("Training set:", nrow(train_data), "observations\n")
cat("Test set:", nrow(test_data), "observations\n")

In [ ]:
# Separate X and y
# Outcome variable is 'poverty'
X_train <- train_data %>% select(-poverty) %>% as.matrix()
y_train <- train_data$poverty

X_test <- test_data %>% select(-poverty) %>% as.matrix()
y_test <- test_data$poverty

In [ ]:
# Normalize features (standardization)
preproc <- preProcess(X_train, method = c("center", "scale"))
X_train_scaled <- predict(preproc, X_train)
X_test_scaled <- predict(preproc, X_test)

## 3. Neural Network Model

In [ ]:
# Define neural network architecture
# Simple feedforward network with 2 hidden layers
nn_model <- keras_model_sequential() %>%
  layer_dense(units = 32, activation = "relu", input_shape = ncol(X_train_scaled)) %>%
  layer_dropout(rate = 0.2) %>%
  layer_dense(units = 16, activation = "relu") %>%
  layer_dropout(rate = 0.2) %>%
  layer_dense(units = 1)  # Output layer for regression

summary(nn_model)

In [ ]:
# Compile model
nn_model %>% compile(
  loss = "mse",
  optimizer = optimizer_adam(learning_rate = 0.001),
  metrics = c("mae")
)

In [ ]:
# Train model
history <- nn_model %>% fit(
  X_train_scaled, y_train,
  epochs = 100,
  batch_size = 8,
  validation_split = 0.2,
  verbose = 1
)

plot(history)

## 4. Model Evaluation

In [ ]:
# Neural Network predictions
nn_pred <- nn_model %>% predict(X_test_scaled)
nn_pred <- as.vector(nn_pred)

# Calculate performance metrics
nn_rmse <- RMSE(nn_pred, y_test)
nn_mae <- MAE(nn_pred, y_test)
nn_r2 <- R2_Score(nn_pred, y_test)

cat("\n=== Neural Network Performance ===\n")
cat("RMSE:", round(nn_rmse, 4), "\n")
cat("MAE:", round(nn_mae, 4), "\n")
cat("R²:", round(nn_r2, 4), "\n")

## 5. Random Forest for Comparison

In [ ]:
# Train Random Forest model (as in Somalia study)
set.seed(2026)
rf_model <- randomForest(
  x = X_train,
  y = y_train,
  ntree = 500,
  importance = TRUE
)

print(rf_model)

In [ ]:
# Random Forest predictions
rf_pred <- predict(rf_model, X_test)

# Calculate performance metrics
rf_rmse <- RMSE(rf_pred, y_test)
rf_mae <- MAE(rf_pred, y_test)
rf_r2 <- R2_Score(rf_pred, y_test)

cat("\n=== Random Forest Performance ===\n")
cat("RMSE:", round(rf_rmse, 4), "\n")
cat("MAE:", round(rf_mae, 4), "\n")
cat("R²:", round(rf_r2, 4), "\n")

## 6. Feature Importance Analysis

In [ ]:
# Extract feature importance from Random Forest
importance_df <- as.data.frame(importance(rf_model)) %>%
  rownames_to_column("Feature") %>%
  arrange(desc(`%IncMSE`))

# Display top 10 features
cat("\n=== Top 10 Most Important Features ===\n")
print(head(importance_df, 10))

In [ ]:
# Plot feature importance
importance_df %>%
  top_n(10, `%IncMSE`) %>%
  ggplot(aes(x = reorder(Feature, `%IncMSE`), y = `%IncMSE`)) +
  geom_col(fill = "steelblue") +
  coord_flip() +
  labs(
    title = "Top 10 Features by Importance (Random Forest)",
    x = "Feature",
    y = "% Increase in MSE"
  ) +
  theme_minimal()

## 7. Model Comparison

In [ ]:
# Compare all models
results <- data.frame(
  Model = c("Neural Network", "Random Forest"),
  RMSE = c(nn_rmse, rf_rmse),
  MAE = c(nn_mae, rf_mae),
  R2 = c(nn_r2, rf_r2)
)

cat("\n=== Model Comparison ===\n")
print(results)

In [ ]:
# Load gridExtra for multi-panel plots
library(gridExtra)

# Visualization: Predicted vs Actual
comparison_df <- data.frame(
  Actual = y_test,
  NN_Predicted = nn_pred,
  RF_Predicted = rf_pred
)

# Neural Network
p1 <- ggplot(comparison_df, aes(x = Actual, y = NN_Predicted)) +
  geom_point(color = "blue", size = 3, alpha = 0.6) +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") +
  labs(title = "Neural Network: Predicted vs Actual",
       x = "Actual Poverty Rate", y = "Predicted Poverty Rate") +
  theme_minimal()

# Random Forest
p2 <- ggplot(comparison_df, aes(x = Actual, y = RF_Predicted)) +
  geom_point(color = "darkgreen", size = 3, alpha = 0.6) +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") +
  labs(title = "Random Forest: Predicted vs Actual",
       x = "Actual Poverty Rate", y = "Predicted Poverty Rate") +
  theme_minimal()

grid.arrange(p1, p2, ncol = 2)

## 8. Conclusions

**Key Findings:**
1. Compare model performance (RMSE, R², MAE)
2. Identify top poverty predictors from feature importance
3. Assess whether deep learning outperforms traditional ML for this small dataset (n=50)

**Limitations:**
- Small sample size (n=50) limits deep learning effectiveness
- Cross-sectional data prevents causal inference
- State-level aggregation (ecological fallacy)

**Next Steps:**
- Compare with OLS and LASSO results from other analyses
- Test alternative architectures (deeper networks, different activations)
- Incorporate findings into final presentation